# Fine-tuning and Alignment — Exercises

Companion to the note [Fine-tuning and Alignment](Fine-tuning%20and%20Alignment.md).

Practise parameter-efficient fine-tuning and preference alignment: LoRA parameter and memory counts, why low-rank updates work, the Bradley–Terry reward model, the KL-regularised RLHF objective and its closed-form optimum, and the DPO loss and its gradient. Everything is done by hand or with NumPy toys (no deep-learning framework needed).

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

rng = np.random.default_rng(0)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

## Part A · Concepts

### Exercise 1 · Picking a fine-tuning method
*🧠 Concept · ★☆☆*

Using the note's table, choose a method for each scenario and justify it:
(a) one 24 GB GPU, adapt a 13B model to a company's support-ticket style;
(b) serve 50 customer-specific variants of one base model from a single GPU;
(c) a tiny budget: adapt a frozen model to a classification task with a few hundred examples;
(d) ship a phone-sized model that mimics a large model's answers.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about trainable parameters, memory for the frozen base, and how many copies of the weights must be stored.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **QLoRA**: the 4-bit base of a 13B model needs ~6.5 GB, the LoRA adapters and their optimizer state are tiny; full fine-tuning would need >200 GB.
(b) **LoRA** adapters: one shared base in memory plus 50 adapters of a few MB each, swapped per request (or batched, as in multi-LoRA serving).
(c) **Prompt / prefix tuning** (or LoRA with small rank): only a few thousand parameters, little risk of overfitting a small dataset.
(d) **Distillation**: train the small model on the large model's outputs (soft labels or generated text).

</details>

### Exercise 2 · The alignment pipeline
*🧠 Concept · ★★☆*

Describe the three stages of InstructGPT-style alignment (SFT → reward model → PPO). For each stage state the data it needs and
the loss it optimises. Then explain what DPO removes from this pipeline and why that is attractive.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The reward model is trained on *pairs* $(y_w, y_l)$ of responses to the same prompt.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **SFT**: (prompt, good response) pairs written by humans; loss = next-token cross-entropy on the response tokens.
2. **Reward model**: prompts with two (or more) sampled responses ranked by humans; Bradley–Terry loss $-\log\sigma(r(x,y_w) - r(x,y_l))$.
3. **PPO**: prompts only; maximise $\mathbb E[r(x,y)] - \beta\,\mathrm{KL}(\pi_\theta\,\|\,\pi_{ref})$ by on-policy RL
([[Q-Learning and Policy Gradients]]), sampling from the policy during training.

**DPO** removes the separate reward model *and* the RL loop: it shows that the KL-regularised optimum has a closed form, so the
reward can be written in terms of the policy itself, which turns preference learning into a simple classification-style loss on
fixed preference pairs. Advantages: one model plus a frozen reference, no sampling during training, far fewer hyperparameters,
more stable. Trade-off: it is offline (no exploration of new responses).

</details>

### Exercise 3 · Why the KL penalty?
*🧠 Concept · ★★☆*

In RLHF the policy maximises $r_\phi(x,y) - \beta\log\frac{\pi_\theta(y\mid x)}{\pi_{ref}(y\mid x)}$. What goes wrong with
$\beta = 0$? What goes wrong with very large $\beta$? Relate your answer to **reward hacking** and **catastrophic forgetting**.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The reward model is only accurate near the responses it was trained on.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $\beta=0$ the policy exploits the learned reward model wherever it is wrong: it drifts to strange outputs (very long, flattering,
repetitive, or adversarial text) that score high but are bad (**reward hacking** / over-optimisation, Goodhart's law). Drifting far
from the SFT model also erodes fluency and general skills (**forgetting**) and collapses diversity.
With very large $\beta$ the policy stays almost equal to $\pi_{ref}$ and learns nothing from the preferences. $\beta$ sets a trust
region around the reference model; the same $\beta$ appears in the DPO loss.

</details>

### Exercise 4 · LoRA initialisation and merging
*🧠 Concept · ★☆☆*

LoRA uses $W = W_0 + \frac{\alpha}{r}BA$ with $B\in\mathbb R^{d\times r}$ initialised to **zero** and $A\in\mathbb R^{r\times k}$ random.
(a) Why is $B = 0$ a good initialisation? Why not initialise *both* to zero?
(b) Why does LoRA add **no** inference latency once training is done?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What is the model's output at step 0? What is the gradient with respect to $A$ if $B = 0$, and with respect to $B$ if $A = 0$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $BA = 0$ at the start, so training begins exactly at the pre-trained model (no random perturbation of a good function).
The gradients are $\nabla_B = G A^\top$ and $\nabla_A = B^\top G$ (with $G = \nabla_W L$). With $A$ random, $\nabla_B \neq 0$ and $B$ starts to move,
after which $A$ also gets gradient. If both were zero, both gradients would be zero forever (a saddle point).
(b) After training you can **merge**: $W \leftarrow W_0 + \frac{\alpha}{r}BA$, a matrix of the original shape, so inference is identical in cost to the
base model (or keep adapters separate to swap them).

</details>

### Exercise 5 · Distillation with soft targets
*🧠 Concept · ★★☆*

In knowledge distillation the student minimises $\mathrm{KL}\big(\text{softmax}(z_t/\tau)\,\|\,\text{softmax}(z_s/\tau)\big)$ between
teacher and student logits at a temperature $\tau > 1$, often combined with the usual hard-label loss.
Why are soft targets more informative than the teacher's argmax label? What does $\tau$ do? How does "sequence-level" distillation
of LLMs differ?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A teacher that says 'cat 0.7, dog 0.25, car 0.05' tells you more than 'cat'.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Soft targets carry **dark knowledge**: which wrong answers are plausible (dog is similar to cat, car is not), giving a richer, lower-variance
training signal per example. A higher $\tau$ flattens both distributions, amplifying the small probabilities so that this similarity
structure actually influences the loss (the gradient is scaled by $1/\tau^2$, hence the usual $\tau^2$ factor).
For LLMs, sequence-level distillation simply fine-tunes the student (SFT) on **text generated** by the teacher; token-level distillation
matches the full next-token distributions when the teacher's logits are available.

</details>

## Part B · By hand

### Exercise 6 · LoRA parameter count
*✍️ By hand · ★☆☆*

A weight matrix has $d = k = 4096$. How many trainable parameters does full fine-tuning of it have, and how many does LoRA with
rank $r=8$ have? What fraction is that?

In [ ]:
full_params = None
lora_params = None
lora_fraction = None

check('full', full_params, 4096 * 4096)
check('LoRA', lora_params, 8 * (4096 + 4096))
check('fraction', lora_fraction, 8 * 8192 / 4096**2)

<details>
<summary><b>💡 Hint</b></summary>

$B$ is $d\times r$ and $A$ is $r\times k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Full: $4096^2 = 16{,}777{,}216$. LoRA: $r(d+k) = 8\cdot8192 = 65{,}536$. Fraction $= 65536/16777216 = 1/256 \approx 0.39\%$.
In general the ratio is $r(d+k)/(dk) = 2r/d$ for square matrices.

```python
full_params = 4096**2
lora_params = 8 * (4096 + 4096)
lora_fraction = lora_params / full_params
```

</details>

### Exercise 7 · LoRA on a whole model
*✍️ By hand · ★★☆*

A 7B model has 32 layers with $d = 4096$. LoRA with $r = 16$ is applied to the query and value projections ($W_Q, W_V$, each $4096\times4096$)
of every layer. How many trainable parameters is that, and how many MB does the adapter take in fp16?

In [ ]:
lora_total = None
adapter_mb = None   # megabytes, 1 MB = 1e6 bytes

check('trainable params', lora_total, 32 * 2 * 16 * (4096 + 4096))
check('adapter size (MB)', adapter_mb, 32 * 2 * 16 * 8192 * 2 / 1e6)

<details>
<summary><b>💡 Hint</b></summary>

Per adapted matrix: $r(d+k)$. Two matrices per layer, 32 layers, 2 bytes per parameter.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$32\cdot2\cdot16\cdot8192 = 8{,}388{,}608$ parameters ($\approx 0.12\%$ of 7B). In fp16 that is $16.8$ MB, versus 14 GB for a full copy
of the model, so storing hundreds of task adapters is cheap.

```python
lora_total = 32 * 2 * 16 * 8192
adapter_mb = lora_total * 2 / 1e6
```

</details>

### Exercise 8 · Memory: full fine-tuning vs QLoRA
*✍️ By hand · ★★☆*

Rule of thumb for full fine-tuning with Adam in mixed precision: 16 bytes per parameter (fp16 weights 2 + fp16 gradients 2 +
fp32 master weights 4 + two fp32 Adam moments 8), ignoring activations. Compute the memory for a 7B model.
With QLoRA the frozen base is stored in 4 bits: how much do the base weights take?

In [ ]:
full_ft_gb = None
qlora_base_gb = None   # 1 GB = 1e9 bytes

check('full fine-tuning (GB)', full_ft_gb, 7e9 * (2 + 2 + 4 + 8) / 1e9)
check('QLoRA base (GB)', qlora_base_gb, 7e9 * 0.5 / 1e9)

<details>
<summary><b>💡 Hint</b></summary>

4 bits = 0.5 bytes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Full fine-tuning: $7\times10^9\cdot16 = 112$ GB (several 80 GB GPUs). QLoRA base: $3.5$ GB; the LoRA parameters (tens of MB) and their Adam
states are negligible, so a 7B (even a 33B, ~17 GB) model can be fine-tuned on one consumer GPU. Gradients still flow *through* the
dequantised base weights, but are not stored for them.

```python
full_ft_gb = 112.0
qlora_base_gb = 3.5
```

</details>

### Exercise 9 · Bradley–Terry reward model
*✍️ By hand · ★☆☆*

The reward model scores the preferred response $r(x,y_w) = 2.0$ and the rejected one $r(x,y_l) = 0.5$.
Under Bradley–Terry, $P(y_w \succ y_l) = \sigma(r_w - r_l)$. Compute this probability and the loss $-\log\sigma(r_w - r_l)$.
What happens to the probability if you add 10 to both rewards?

In [ ]:
p_prefer = None
bt_loss = None

check('P(y_w > y_l)', p_prefer, sigmoid(1.5))
check('loss', bt_loss, -np.log(sigmoid(1.5)))

<details>
<summary><b>💡 Hint</b></summary>

$\sigma(1.5) = 1/(1+e^{-1.5})$, and $e^{-1.5}\approx 0.2231$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sigma(1.5) = 1/1.2231 \approx 0.818$; loss $= -\ln 0.818 \approx 0.201$. Adding a constant to both rewards changes nothing: only
**differences** of rewards are identified by pairwise preferences. That shift invariance is exactly what lets DPO cancel the
intractable partition function $Z(x)$.

```python
p_prefer = 1 / (1 + np.exp(-1.5))
bt_loss = np.log(1 + np.exp(-1.5))
```

</details>

### Exercise 10 · DPO loss by hand
*✍️ By hand · ★★☆*

With $\beta = 0.1$: the policy gives $\log\pi_\theta(y_w\mid x) = -10$, $\log\pi_\theta(y_l\mid x) = -15$; the reference gives
$\log\pi_{ref}(y_w\mid x) = -12$, $\log\pi_{ref}(y_l\mid x) = -14$. Compute the implicit reward margin
$m = \beta\big[\log\frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}\big]$, the DPO loss
$-\log\sigma(m)$, and the derivative $\partial L/\partial m$.

In [ ]:
dpo_margin = None
dpo_loss = None
dL_dm = None

check('margin', dpo_margin, 0.1 * ((-10 + 12) - (-15 + 14)))
check('loss', dpo_loss, np.logaddexp(0, -0.3))
check('dL/dm', dL_dm, -sigmoid(-0.3))

<details>
<summary><b>💡 Hint 1</b></summary>

Log-ratios: $-10-(-12) = 2$ for the winner, $-15-(-14) = -1$ for the loser.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\frac{d}{dm}\big(-\log\sigma(m)\big) = -(1-\sigma(m)) = -\sigma(-m)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$m = 0.1\,(2 - (-1)) = 0.3$. Loss $= \ln(1+e^{-0.3}) \approx 0.554$. $\partial L/\partial m = -\sigma(-0.3) \approx -0.426$.
The gradient magnitude $\sigma(-m)$ is large when the model **wrongly** prefers $y_l$ ($m<0$) and vanishes when the pair is already
ranked correctly with a large margin: DPO automatically focuses on the pairs it gets wrong.

```python
dpo_margin = 0.3
dpo_loss = np.log(1 + np.exp(-0.3))
dL_dm = -1 / (1 + np.exp(0.3))
```

</details>

### Exercise 11 · The closed-form RLHF optimum
*✍️ By hand · ★★★*

For a fixed prompt, maximise over distributions $\pi$
$$J(\pi) = \sum_y \pi(y)\,r(y) - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{ref}).$$
Show that the maximiser is $\pi^*(y) = \frac{1}{Z}\pi_{ref}(y)\,e^{r(y)/\beta}$ with $Z = \sum_y\pi_{ref}(y)e^{r(y)/\beta}$, that
$J(\pi^*) = \beta\log Z$, and rearrange to express $r(y)$ in terms of $\pi^*$. Explain how this gives the DPO loss.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Write $J(\pi) = -\beta\sum_y \pi(y)\log\frac{\pi(y)}{\pi_{ref}(y)e^{r(y)/\beta}} = -\beta\,\mathrm{KL}(\pi\,\|\,\pi^*) + \beta\log Z$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

KL is $\ge 0$ with equality iff the distributions are equal.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$J(\pi) = \sum_y\pi(y)\big[r(y) - \beta\log\frac{\pi(y)}{\pi_{ref}(y)}\big] = -\beta\sum_y\pi(y)\log\frac{\pi(y)}{\pi_{ref}(y)e^{r(y)/\beta}}
= -\beta\,\mathrm{KL}(\pi\,\|\,\pi^*) + \beta\log Z$. Since KL $\ge 0$, the maximum is at $\pi = \pi^*$ with value $\beta\log Z$.

Rearranging: $r(y) = \beta\log\frac{\pi^*(y)}{\pi_{ref}(y)} + \beta\log Z$. Plug this into the Bradley–Terry likelihood
$\sigma(r(y_w) - r(y_l))$: the $\beta\log Z$ terms **cancel** (same prompt), leaving
$\sigma\big(\beta\log\frac{\pi(y_w)}{\pi_{ref}(y_w)} - \beta\log\frac{\pi(y_l)}{\pi_{ref}(y_l)}\big)$. Maximising its log-likelihood over the
policy parameters is the DPO loss from the note: the policy *is* the reward model.

</details>

## Part C · Code with NumPy toys

### Exercise 12 · LoRA forward pass and rank
*💻 Code · ★☆☆*

Implement `lora_forward(x, W0, A, B, alpha, r)` computing $y = x W_0^\top + \frac{\alpha}{r}\,x A^\top B^\top$ (rows of `x` are inputs)
**without** forming $BA$, and `lora_param_count(d, k, r)`. Verify that the result equals the merged-weight computation
$x(W_0 + \frac{\alpha}{r}BA)^\top$ and that $\operatorname{rank}(BA)\le r$.

In [ ]:
def lora_forward(x, W0, A, B, alpha, r):
    # TODO
    return None

def lora_param_count(d, k, r):
    # TODO
    return None

d_, k_, r_ = 64, 48, 4
W0_ = rng.normal(size=(d_, k_)); A_ = rng.normal(size=(r_, k_)); B_ = rng.normal(size=(d_, r_))
x_ = rng.normal(size=(10, k_))
y_lora = lora_forward(x_, W0_, A_, B_, alpha=8, r=r_)

check('equals merged weights', y_lora, x_ @ (W0_ + 8 / r_ * B_ @ A_).T)
check('param count', lora_param_count(4096, 4096, 8), 65536)
check('rank of BA', np.linalg.matrix_rank(B_ @ A_), r_)

<details>
<summary><b>💡 Hint</b></summary>

`x @ W0.T + (alpha / r) * (x @ A.T) @ B.T`: the second term costs $O(r(d+k))$ per input instead of $O(dk)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Computing $(xA^\top)B^\top$ never forms the $d\times k$ update during training, which saves memory. At deployment you merge once.
$BA$ is a product through an $r$-dimensional bottleneck, so its rank is at most $r$ (here exactly 4).

```python
def lora_forward(x, W0, A, B, alpha, r):
    return x @ W0.T + (alpha / r) * (x @ A.T) @ B.T

def lora_param_count(d, k, r):
    return r * (d + k)

d_, k_, r_ = 64, 48, 4
W0_ = rng.normal(size=(d_, k_)); A_ = rng.normal(size=(r_, k_)); B_ = rng.normal(size=(d_, r_))
x_ = rng.normal(size=(10, k_))
y_lora = lora_forward(x_, W0_, A_, B_, alpha=8, r=r_)
```

</details>

### Exercise 13 · Best rank-r update via SVD
*💻 Code · ★★☆*

Suppose the "ideal" fine-tuning update $\Delta W$ (here a random $50\times40$ matrix whose spectrum decays quickly) were known.
The best rank-$r$ approximation in Frobenius norm is the truncated SVD (Eckart–Young). Implement `best_rank_r(M, r)` and store the
relative error $\lVert \Delta W - \Delta W_r\rVert_F / \lVert\Delta W\rVert_F$ for $r = 1, 2, 4, 8, 16$ in `rel_err` (a list).
What does a fast-decaying spectrum mean for LoRA?

In [ ]:
U0, _ = np.linalg.qr(rng.normal(size=(50, 40))); V0, _ = np.linalg.qr(rng.normal(size=(40, 40)))
dW_ideal = U0 @ np.diag(2.0 ** -np.arange(40)) @ V0.T * 10     # singular values 10, 5, 2.5, ...

def best_rank_r(M, r):
    # TODO
    return None

rel_err = None

_s = np.linalg.svd(dW_ideal, compute_uv=False)
_ref = [np.sqrt((_s[r:] ** 2).sum() / (_s ** 2).sum()) for r in (1, 2, 4, 8, 16)]
check('relative errors', rel_err, _ref)

<details>
<summary><b>💡 Hint</b></summary>

`U, s, Vt = np.linalg.svd(M, full_matrices=False)`; keep `U[:, :r] * s[:r] @ Vt[:r]`. Then $B = U_r\Sigma_r$, $A = V_r^\top$ is a LoRA factorisation.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The error is $\sqrt{\sum_{i>r}s_i^2/\sum_i s_i^2}$: about $0.5, 0.25, 0.06, 0.004, 2\times10^{-5}$. Hu et al. observed that fine-tuning
updates have low **intrinsic rank**, i.e. a quickly decaying spectrum, so small $r$ (4–16) captures almost all of the update; that is the
whole bet behind LoRA. (PiSSA/related methods even initialise $A,B$ from the SVD of $W_0$.)

```python
U0, _ = np.linalg.qr(rng.normal(size=(50, 40))); V0, _ = np.linalg.qr(rng.normal(size=(40, 40)))
dW_ideal = U0 @ np.diag(2.0 ** -np.arange(40)) @ V0.T * 10

def best_rank_r(M, r):
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    return (U[:, :r] * s[:r]) @ Vt[:r]

rel_err = [np.linalg.norm(dW_ideal - best_rank_r(dW_ideal, r)) / np.linalg.norm(dW_ideal) for r in (1, 2, 4, 8, 16)]
print(np.round(rel_err, 5))
```

</details>

### Exercise 14 · Training a LoRA adapter by gradient descent
*💻 Code · ★★★*

A frozen layer $W_0$ ($20\times20$) must be adapted to a new task whose true weights are $W_0 + \Delta W$ with $\operatorname{rank}(\Delta W)=2$.
Train only $A\in\mathbb R^{2\times20}$ (init $\mathcal N(0, 0.01^2)$) and $B\in\mathbb R^{20\times2}$ (init 0) by full-batch gradient descent on
$L = \frac{1}{2N}\sum_n\lVert (W_0 + BA)x_n - y_n\rVert^2$ (lr 0.05, 3000 steps). Store the final relative error
$\lVert BA - \Delta W\rVert_F/\lVert\Delta W\rVert_F$ in `lora_rel_err`. Derive $\nabla_A L$ and $\nabla_B L$ first.

In [ ]:
r2 = np.random.default_rng(1)
W0_task = r2.normal(size=(20, 20)) / np.sqrt(20)
dW_true = r2.normal(size=(20, 2)) @ r2.normal(size=(2, 20)) / np.sqrt(20)
X_task = r2.normal(size=(500, 20))
Y_task = X_task @ (W0_task + dW_true).T
lora_rel_err = None

check('adapter recovers the low-rank update', None if lora_rel_err is None else lora_rel_err < 1e-3, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Let $G = \nabla_W L = \frac1N E^\top X$ with residuals $E = X(W_0+BA)^\top - Y$. By the chain rule for $W = W_0 + BA$: $\nabla_B L = G A^\top$, $\nabla_A L = B^\top G$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Compute both gradients from the same $G$ before updating either matrix.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $G = \frac1N E^\top X$: $\nabla_B L = GA^\top$ and $\nabla_A L = B^\top G$. Starting from $B=0$, the first steps only move $B$; then both move,
and since the target update really has rank 2, the adapter recovers it essentially exactly (relative error $\sim10^{-15}$) while
training only $2\cdot2\cdot20 = 80$ of the 400 weights. If the true update had rank 5, a rank-2 adapter could at best reach the
truncated-SVD error from the previous exercise.

```python
r2 = np.random.default_rng(1)
W0_task = r2.normal(size=(20, 20)) / np.sqrt(20)
dW_true = r2.normal(size=(20, 2)) @ r2.normal(size=(2, 20)) / np.sqrt(20)
X_task = r2.normal(size=(500, 20))
Y_task = X_task @ (W0_task + dW_true).T

A_t = r2.normal(size=(2, 20)) * 0.01
B_t = np.zeros((20, 2))
for _ in range(3000):
    E = X_task @ (W0_task + B_t @ A_t).T - Y_task
    G = E.T @ X_task / len(X_task)
    gB, gA = G @ A_t.T, B_t.T @ G
    B_t -= 0.05 * gB
    A_t -= 0.05 * gA
lora_rel_err = np.linalg.norm(B_t @ A_t - dW_true) / np.linalg.norm(dW_true)
print(lora_rel_err)
```

</details>

### Exercise 15 · A vectorised DPO loss
*💻 Code · ★☆☆*

Implement `dpo_loss(pol_w, pol_l, ref_w, ref_l, beta)` taking arrays of sequence log-probabilities for a batch of preference pairs and
returning the **mean** loss, computed stably with `np.logaddexp` (note $-\log\sigma(m) = \log(1+e^{-m})$). Also return the
"reward accuracy" (fraction of pairs with margin $> 0$), a metric that DPO training logs report.

In [ ]:
def dpo_loss(pol_w, pol_l, ref_w, ref_l, beta=0.1):
    # TODO: return (mean_loss, reward_accuracy)
    return None

dpo_out = dpo_loss(np.array([-10., -20, -5]), np.array([-15., -18, -9]),
                   np.array([-12., -19, -6]), np.array([-14., -19, -8]), beta=0.1)

_m = 0.1 * (np.array([2., -1, 1]) - np.array([-1., 1, -1]))
check('mean DPO loss', None if dpo_out is None else dpo_out[0], np.mean(-np.log(sigmoid(_m))))
check('reward accuracy', None if dpo_out is None else dpo_out[1], 2 / 3)

<details>
<summary><b>💡 Hint</b></summary>

`m = beta * ((pol_w - ref_w) - (pol_l - ref_l))`; `np.logaddexp(0, -m)` is $\log(1+e^{-m})$ without overflow.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Margins: $0.3, -0.2, 0.2$ → losses $0.554, 0.798, 0.598$, mean $\approx 0.650$, reward accuracy $2/3$. The first pair is the
by-hand example above. `logaddexp` stays finite even for margins of $\pm1000$, where `-np.log(sigmoid(m))` would give `inf`.

```python
def dpo_loss(pol_w, pol_l, ref_w, ref_l, beta=0.1):
    m = beta * ((pol_w - ref_w) - (pol_l - ref_l))
    return np.mean(np.logaddexp(0, -m)), np.mean(m > 0)

dpo_out = dpo_loss(np.array([-10., -20, -5]), np.array([-15., -18, -9]),
                   np.array([-12., -19, -6]), np.array([-14., -19, -8]), beta=0.1)
```

</details>

### Exercise 16 · KL-regularised optimum, numerically
*💻 Code · ★★☆*

For 5 candidate responses with rewards `r5` and reference policy `ref5`, implement `rlhf_objective(pi, r, ref, beta)`
$= \sum_y \pi(y)r(y) - \beta\,\mathrm{KL}(\pi\|\pi_{ref})$ and `optimal_policy(r, ref, beta)` $= \pi_{ref}e^{r/\beta}/Z$.
Check numerically that no random policy (Dirichlet samples) beats $\pi^*$ and that $J(\pi^*) = \beta\log Z$. Then look at $\pi^*$ for
$\beta = 10, 1, 0.1$: what happens?

In [ ]:
r5 = np.array([1.0, 2.0, 0.5, 3.0, 0.0])
ref5 = np.array([0.4, 0.3, 0.15, 0.05, 0.1])

def rlhf_objective(pi, r, ref, beta):
    # TODO
    return None

def optimal_policy(r, ref, beta):
    # TODO
    return None

pi_star = optimal_policy(r5, ref5, beta=1.0)

if pi_star is not None:
    _J = lambda p: p @ r5 - 1.0 * np.sum(p * np.log(p / ref5))
    _rand = rng.dirichlet(np.ones(5), size=2000)
    check('no random policy beats pi*', _J(np.asarray(pi_star)) >= max(_J(p) for p in _rand), True)
    check('J(pi*) = beta log Z', rlhf_objective(pi_star, r5, ref5, 1.0), np.log(ref5 @ np.exp(r5)))
else:
    check('no random policy beats pi*', None, True)

<details>
<summary><b>💡 Hint</b></summary>

KL$(\pi\|\pi_{ref}) = \sum_y\pi(y)\log\frac{\pi(y)}{\pi_{ref}(y)}$; normalise with `w / w.sum()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\beta=10$: $\pi^*\approx\pi_{ref}$ (the KL term dominates). $\beta = 1$: mass shifts towards high-reward responses, but response 3 (reward 3,
reference probability only 0.05) does not take over completely. $\beta=0.1$: $\pi^*$ is almost a point mass on the argmax reward,
i.e. pure reward maximisation, with all the reward-hacking risks discussed in Part A.

```python
r5 = np.array([1.0, 2.0, 0.5, 3.0, 0.0])
ref5 = np.array([0.4, 0.3, 0.15, 0.05, 0.1])

def rlhf_objective(pi, r, ref, beta):
    pi = np.asarray(pi, float)
    return pi @ r - beta * np.sum(pi * np.log(pi / ref))

def optimal_policy(r, ref, beta):
    w = ref * np.exp((r - r.max()) / beta)
    return w / w.sum()

pi_star = optimal_policy(r5, ref5, beta=1.0)
for b in (10, 1, 0.1):
    print(b, np.round(optimal_policy(r5, ref5, b), 3))
```

</details>

### Exercise 17 · DPO on a toy policy
*💻 Code · ★★★*

A "policy" over 4 possible responses is $\pi_\theta = \text{softmax}(\theta)$, the reference is uniform, and humans prefer responses by
the hidden reward `r_hidden`. Build all preference pairs $(y_w, y_l)$ with $r(y_w) > r(y_l)$ and minimise the mean DPO loss ($\beta = 0.5$)
by gradient descent on $\theta$ (lr 0.5, 500 steps).

1. Show by hand that $\nabla_\theta L_{pair} = -\beta\,\sigma(-m)\,(e_{y_w} - e_{y_l})$: the log-normaliser of the softmax cancels.
2. Store the learned `theta_dpo` and verify the policy ranks responses like `r_hidden`. Inspect $\pi_\theta$: what do you notice?

In [ ]:
r_hidden = np.array([1.0, 3.0, 0.0, 2.0])
ref_toy = np.full(4, 0.25)
theta_dpo = None

check('policy ranks responses by hidden reward', None if theta_dpo is None else list(np.argsort(-np.asarray(theta_dpo))) == list(np.argsort(-r_hidden)), True)

<details>
<summary><b>💡 Hint 1</b></summary>

$\log\pi_\theta(y) = \theta_y - \log\sum_j e^{\theta_j}$: the second term is identical for $y_w$ and $y_l$, so it cancels in the margin.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Accumulate the per-pair gradients into a length-4 vector and divide by the number of pairs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $m = \beta[(\theta_w - \log Z_\theta - \log\pi_{ref,w}) - (\theta_l - \log Z_\theta - \log\pi_{ref,l})] = \beta(\theta_w-\theta_l) + \text{const}$, so
$\partial L/\partial\theta = -\sigma(-m)\,\beta\,(e_w - e_l)$.
2. The learned $\theta$ ranks the responses $1 \succ 3 \succ 0 \succ 2$, matching the hidden reward. But $\pi_\theta$ ends up putting ~99 % of the mass on the
single best response: with deterministic, noise-free preferences the DPO optimum pushes margins to infinity, so the policy drifts far
from the reference. This over-optimisation is a known DPO failure mode; fixes include label smoothing (conservative DPO), IPO, early stopping
and monitoring the KL to the reference.

```python
r_hidden = np.array([1.0, 3.0, 0.0, 2.0])
ref_toy = np.full(4, 0.25)
beta = 0.5
pairs = [(w, l) for w in range(4) for l in range(4) if r_hidden[w] > r_hidden[l]]
theta_dpo = np.zeros(4)
for _ in range(500):
    logp = theta_dpo - np.log(np.exp(theta_dpo).sum())
    g = np.zeros(4)
    for w, l in pairs:
        m = beta * ((logp[w] - np.log(ref_toy[w])) - (logp[l] - np.log(ref_toy[l])))
        s = sigmoid(-m)
        g[w] -= beta * s
        g[l] += beta * s
    theta_dpo -= 0.5 * g / len(pairs)
print(np.round(np.exp(theta_dpo) / np.exp(theta_dpo).sum(), 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Fine-tuning and Alignment](Fine-tuning%20and%20Alignment.md).*